# Step 2 - Hyperparameter tuning

Compare the baseline with 20 randomly sampled XGBoost configurations using three chronological folds within the training split. Select by mean fold average precision, refit on the original full training split, and evaluate on validation. Random undersampling remains 0.1, class weighting remains 1 and the decision threshold remains 0.5.

Candidate 15 achieved the highest mean fold AP, 0.679166, compared with 0.653863 for the baseline configuration. The selected model's validation AP was 0.709403, compared with 0.705831 for the fitted baseline.

The validation set previously influenced model-family selection, so its results are development estimates. The upstream outer split boundaries were selected using label rates across periods, including test; this limitation remains. Test data are not loaded in this notebook.


The saved notebook snapshot contains an interrupted rerun during training-file hashing. Completed-run results are available in the exported result files and are displayed in the saved-results section below. The interruption traceback is omitted; no model fitting was rerun to produce this view.


## 1. Environment and paths

Use the same package versions as Step 1. A fresh kernel avoids retaining large dataframes from earlier runs. The configuration checks package versions and loads the saved baseline. Optional Colab mounting is shown in the path configuration.


In [8]:
# Optional installation; retain the recorded package versions:
# %pip install numpy pandas scikit-learn imbalanced-learn xgboost joblib

from pathlib import Path
import gc
import hashlib
import importlib.metadata
import json
import os
import time
import warnings

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import ParameterSampler
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier
from IPython.display import display

# Project root: directory containing these notebooks and output folders.
PROJECT_DIR = Path.cwd()

# Optional Google Colab setup:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = Path('/content/drive/MyDrive/fine tuning')

# CSV splits can be stored in data/ or directly in PROJECT_DIR.
DATA_DIR = PROJECT_DIR / 'data' if (PROJECT_DIR / 'data').is_dir() else PROJECT_DIR
BASELINE_DIR = PROJECT_DIR / 'baseline_outputs'
OUTPUT_DIR = PROJECT_DIR / 'step_2_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_CANDIDATES = 20  # Set to 5 BEFORE the first run for a shorter pilot.
LABEL_DELAY_DAYS = 7  # Keep the feature-engineering assumption; not a tuned parameter.
GAP_DAYS = 7
SAMPLING_RATIO = 0.1
LABEL = 'isFraud'
EXCLUDE = [LABEL, 'TransactionID', 'TransactionDT', 'TransactionDateTime', 'Transaction_Day', 'uid']

def jsonable(value):
    return json.loads(json.dumps(value, default=lambda v: v.item() if isinstance(v, np.generic) else str(v)))

def save_json(path, value):
    Path(path).write_text(json.dumps(jsonable(value), indent=2), encoding='utf-8')

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def locate_one(names):
    found = [DATA_DIR / n for n in names if (DATA_DIR / n).is_file()]
    if len(found) != 1:
        raise ValueError(f'Expected exactly one of {names} in {DATA_DIR}; found {found}')
    return found[0]

TRAIN_PATH = locate_one(['train.csv', 'train (2).csv'])
VAL_PATH = locate_one(['val.csv'])
run_path = BASELINE_DIR / 'baseline_run.json'
bundle_path = BASELINE_DIR / 'baseline_bundle.joblib'
assert run_path.is_file() and bundle_path.is_file(), 'Run Step 1 completely first; baseline outputs are missing.'
baseline_run = json.loads(run_path.read_text())
package_names = ['numpy', 'pandas', 'scikit-learn', 'imbalanced-learn', 'xgboost', 'joblib']
runtime_versions = {name: importlib.metadata.version(name) for name in package_names}
mismatches = {name: (baseline_run['versions'].get(name), version)
              for name, version in runtime_versions.items()
              if baseline_run['versions'].get(name) != version}
if mismatches:
    raise RuntimeError(f'Package versions differ from Step 1: {mismatches}. Use the Step 1 environment or rerun Step 1 in this environment before tuning.')
# Load the baseline artifact produced by Step 1.
baseline_bundle = joblib.load(bundle_path)
baseline_model = baseline_bundle['model']
FEATURES = baseline_bundle['features']
CAT_COLS = baseline_bundle['categorical_columns']
baseline_encoder = baseline_bundle['encoder']
assert len(FEATURES) == 482 and len(CAT_COLS) == 37
assert baseline_model.get_params()['scale_pos_weight'] == 1.0
print('Working baseline AP:', baseline_run['validation_metrics']['Average precision'])
print('Versions:', runtime_versions)
print('Output directory:', OUTPUT_DIR)

Working baseline AP: 0.7058307163138061
Versions: {'numpy': '2.1.3', 'pandas': '2.2.3', 'scikit-learn': '1.6.1', 'imbalanced-learn': '0.14.2', 'xgboost': '3.4.1', 'joblib': '1.6.0'}


## 2. Training data and chronological folds

Verify the training-file fingerprint against Step 1. Define scoring windows using calendar-day boundaries at approximately 55%, 70%, 85% and 100% of the training period. Each fold fits an earlier prefix, leaves a seven-day gap and scores the following block. Boundaries depend on time rather than fraud rates; scoring folds retain their natural class proportions. The separate validation CSV is not read before hyperparameter selection.


In [ ]:
baseline_audit = json.loads((BASELINE_DIR / 'data_audit.json').read_text())
train_hash = sha256(TRAIN_PATH)
assert train_hash == baseline_audit['input_files']['train']['sha256'], 'Training CSV differs from Step 1.'
raw = pd.read_csv(TRAIN_PATH, low_memory=False)
assert [c for c in raw.columns if c not in EXCLUDE] == FEATURES
assert raw[LABEL].notna().all() and raw[LABEL].isin([0, 1]).all()
assert raw['TransactionDT'].is_monotonic_increasing
assert raw['TransactionID'].is_unique and raw['uid'].notna().all()

def make_folds(frame, gap_days=7):
    t = frame['TransactionDT'].to_numpy(dtype=np.int64)
    days = np.unique(t // 86400)
    boundaries = [int(days[int(len(days) * f)]) * 86400 for f in [0.55, 0.70, 0.85]]
    boundaries.append((int(days[-1]) + 1) * 86400)
    folds, rows = [], []
    for i, (start, end) in enumerate(zip(boundaries[:-1], boundaries[1:]), 1):
        fit_idx = np.flatnonzero(t < start - gap_days * 86400)
        score_idx = np.flatnonzero((t >= start) & (t < end))
        assert len(fit_idx) and len(score_idx)
        assert t[fit_idx].max() < t[score_idx].min()
        assert frame.iloc[fit_idx][LABEL].nunique() == frame.iloc[score_idx][LABEL].nunique() == 2
        folds.append((fit_idx, score_idx))
        rows.append({'fold': i, 'fit_rows': len(fit_idx), 'score_rows': len(score_idx),
                     'fit_end_seconds': int(t[fit_idx].max()), 'score_start_seconds': start,
                     'score_end_seconds_exclusive': end,
                     'fit_fraud_rate': float(frame.iloc[fit_idx][LABEL].mean()),
                     'score_fraud_rate': float(frame.iloc[score_idx][LABEL].mean())})
    return folds, pd.DataFrame(rows)

folds, fold_table = make_folds(raw, GAP_DAYS)
fold_table.to_csv(OUTPUT_DIR / 'fold_summary.csv', index=False)
display(fold_table)

## 3. Fold-specific feature construction

Recompute all 18 frequency columns using each fold's fitting rows before undersampling. Recompute both historical fraud features using only fitting-row labels that are at least seven days old at the scored transaction. Labels from gap and scoring rows cannot contribute.

Fit the categorical encoder on fitting rows only, then apply undersampling. Other engineered features retain the upstream notebook's past-only construction, including earlier unlabeled transaction history. These checks address fold-dependent features and do not constitute a separate audit of every upstream feature.


In [ ]:
FREQ_COLS = [c for c in FEATURES if c.endswith('_freq')]
assert len(FREQ_COLS) == 18
assert all(c[:-5] in raw.columns for c in FREQ_COLS)

def clean_frequency_key(series):
    s = series.astype(object)
    return s.where(s.notna() & ~s.isin(['unknown_device']), '__MISSING__')

def known_fraud_features(frame, fit_idx, delay_days=7):
    # Same composite-key/counting principle as the project's feature notebook.
    n = len(frame)
    codes = pd.factorize(frame['uid'], sort=False)[0].astype(np.int64)
    assert (codes >= 0).all()
    t = frame['TransactionDT'].to_numpy(dtype=np.int64)
    assert (t >= 0).all() and (t < 10**9).all()
    allowed_y = np.zeros(n, dtype=np.int64)
    allowed_y[fit_idx] = frame[LABEL].to_numpy(dtype=np.int64)[fit_idx]
    order = np.argsort(codes, kind='stable')
    c, tt, yy = codes[order], t[order], allowed_y[order]
    key = c * 10**9 + tt
    assert np.all(key[:-1] <= key[1:])
    starts = np.flatnonzero(np.r_[True, c[1:] != c[:-1]])
    group_start = np.repeat(starts, np.diff(np.r_[starts, n]))
    j = np.searchsorted(key, key - int(delay_days * 86400), side='right')
    j = np.maximum(j, group_start)
    prefix = np.r_[0, np.cumsum(yy)]
    count = prefix[j] - prefix[group_start]
    restored = np.empty(n, dtype=np.int32)
    restored[order] = count.astype(np.int32)
    return restored, (restored > 0).astype(np.int8)

def prepare_fold(frame, fit_idx, score_idx, features, freq_cols):
    X_fit = frame.iloc[fit_idx][features].copy()
    X_score = frame.iloc[score_idx][features].copy()
    for output_col in freq_cols:
        source_col = output_col[:-5]
        key_fit = clean_frequency_key(frame[source_col].iloc[fit_idx])
        mapping = key_fit.value_counts(normalize=True)
        X_fit[output_col] = key_fit.map(mapping).fillna(0).to_numpy(dtype=np.float32)
        key_score = clean_frequency_key(frame[source_col].iloc[score_idx])
        X_score[output_col] = key_score.map(mapping).fillna(0).to_numpy(dtype=np.float32)
    counts, flags = known_fraud_features(frame, fit_idx, LABEL_DELAY_DAYS)
    for X, idx in [(X_fit, fit_idx), (X_score, score_idx)]:
        X['prev_uid_fraud_count'] = counts[idx]
        X['prev_uid_has_fraud'] = flags[idx]
        X.replace([np.inf, -np.inf], np.nan, inplace=True)
    cats = X_fit.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    enc = None
    if cats:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X_fit[cats] = enc.fit_transform(X_fit[cats].astype(str))
        X_score[cats] = enc.transform(X_score[cats].astype(str))
    return X_fit, X_score, frame.iloc[fit_idx][LABEL].to_numpy(dtype=int), frame.iloc[score_idx][LABEL].to_numpy(dtype=int)

## 4. Search configuration and checkpoints

Candidate 0 is the baseline. Twenty sampled configurations vary depth, child weight, learning rate, tree count, row and column sampling, and regularization. All candidates use the same sampled fitting data within a fold. Tree count is tuned directly without early stopping on the scoring fold.

The search comprises 63 fits plus a possible full-data refit. Save results after each fit and resume completed folds when the configuration is unchanged. A changed search requires a separate output directory.


In [ ]:
SEARCH_SPACE = {
    'max_depth': [3, 4, 5, 6, 8],
    'min_child_weight': [1, 3, 5, 10],
    'learning_rate': [0.03, 0.05, 0.1, 0.2, 0.3],
    'n_estimators': [100, 200, 400, 700],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.7, 0.85, 1.0],
    'reg_alpha': [0.0, 0.1, 1.0],
    'reg_lambda': [1.0, 3.0, 10.0],
}
candidates = [{}] + jsonable(list(ParameterSampler(SEARCH_SPACE, n_iter=N_CANDIDATES, random_state=SEED)))
manifest = jsonable({'implementation': 'step2_temporal_v1', 'train_sha256': train_hash,
    'baseline_bundle_sha256': sha256(bundle_path), 'versions': runtime_versions,
    'seed': SEED, 'gap_days': GAP_DAYS, 'label_delay_days': LABEL_DELAY_DAYS,
    'sampling_ratio': SAMPLING_RATIO, 'candidates': candidates,
    'folds': fold_table.to_dict('records')})
manifest_path = OUTPUT_DIR / 'search_manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, 'Search inputs changed. Use a new OUTPUT_DIR.'
else:
    save_json(manifest_path, manifest)
checkpoint_path = OUTPUT_DIR / 'cv_fit_results.csv'
records = pd.read_csv(checkpoint_path).to_dict('records') if checkpoint_path.exists() else []
done = {(int(r['fold']), int(r['candidate_id'])) for r in records}
assert len(done) == len(records), 'Duplicate checkpoint rows.'
print(f'{len(candidates)} configurations x {len(folds)} folds; {len(done)} fits already completed.')

for fold_id, (fit_idx, score_idx) in enumerate(folds, 1):
    pending = [i for i in range(len(candidates)) if (fold_id, i) not in done]
    if not pending:
        continue
    print(f'Preparing fold {fold_id}...', flush=True)
    X_fit, X_score, y_fit, y_score = prepare_fold(raw, fit_idx, score_idx, FEATURES, FREQ_COLS)
    sampler = RandomUnderSampler(sampling_strategy=SAMPLING_RATIO, random_state=SEED)
    X_sampled, y_sampled = sampler.fit_resample(X_fit, y_fit)
    del X_fit, y_fit
    gc.collect()
    for candidate_id in pending:
        started = time.perf_counter()
        estimator = clone(baseline_model).set_params(**candidates[candidate_id])
        estimator.fit(X_sampled, y_sampled)
        score = estimator.predict_proba(X_score)[:, 1]
        ap = float(average_precision_score(y_score, score))
        assert np.isfinite(ap)
        record = {'fold': fold_id, 'candidate_id': candidate_id, 'average_precision': ap,
                  'fit_and_score_seconds': time.perf_counter() - started,
                  'fit_rows_after_sampling': len(y_sampled), 'score_rows': len(y_score)}
        records.append(record)
        temp_path = checkpoint_path.with_suffix('.tmp')
        pd.DataFrame(records).to_csv(temp_path, index=False)
        os.replace(temp_path, checkpoint_path)
        done.add((fold_id, candidate_id))
        print(f'Fold {fold_id} | candidate {candidate_id:02d} | AP={ap:.6f} | {record["fit_and_score_seconds"]:.1f}s', flush=True)
        del estimator, score
        gc.collect()
    del X_sampled, y_sampled, X_score, y_score
    gc.collect()

## 5. Configuration selection

Select the configuration with the highest mean fold AP. The fold standard deviation describes variation across time windows; it is not a confidence interval. The baseline remains eligible. Fold scores and the original validation score cover different periods and training sizes and are not directly comparable.


In [ ]:
cv = pd.DataFrame(records)
assert len(cv) == len(candidates) * len(folds)
summary = cv.groupby('candidate_id').agg(
    mean_cv_ap=('average_precision', 'mean'), std_cv_ap=('average_precision', 'std'),
    folds_completed=('fold', 'nunique'), total_seconds=('fit_and_score_seconds', 'sum')).reset_index()
assert (summary['folds_completed'] == len(folds)).all()
summary['parameters'] = summary['candidate_id'].map(lambda i: json.dumps(candidates[int(i)], sort_keys=True))
summary = summary.sort_values(['mean_cv_ap', 'candidate_id'], ascending=[False, True]).reset_index(drop=True)
summary.to_csv(OUTPUT_DIR / 'cv_summary.csv', index=False)
best_id = int(summary.iloc[0]['candidate_id'])
best_params = candidates[best_id]
baseline_cv_ap = float(summary.loc[summary['candidate_id'] == 0, 'mean_cv_ap'].iloc[0])
selection = {'candidate_id': best_id, 'parameters': best_params,
             'mean_cv_ap': float(summary.iloc[0]['mean_cv_ap']),
             'baseline_mean_cv_ap': baseline_cv_ap,
             'mean_cv_ap_gain': float(summary.iloc[0]['mean_cv_ap']) - baseline_cv_ap,
             'selection_metric': 'mean of three temporal-fold average precision scores'}
save_json(OUTPUT_DIR / 'selected_parameters.json', selection)
display(summary.head(10))
print('Selected:', selection)

## 6. Full-training refit and validation comparison

Refit the selected configuration using the original full training split, engineered features, saved encoder and undersampling seed. The seven-day gap applies to internal folds; final refitting retains the original retrospective training protocol.

Verify the saved baseline AP before comparing both models at threshold 0.5. Preserve the configuration selected by cross-validation rather than repeatedly searching against the external validation set.


In [ ]:
# Release the training-only fold dataframe before loading final matrices.
del raw
gc.collect()
assert sha256(VAL_PATH) == baseline_audit['input_files']['val']['sha256'], 'Validation CSV differs from Step 1.'

def baseline_transform(frame, bundle):
    X = frame.loc[:, bundle['features']].copy()
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    cats = bundle['categorical_columns']
    if cats:
        X[cats] = bundle['encoder'].transform(X[cats].astype(str))
    return X

val_frame = pd.read_csv(VAL_PATH, low_memory=False)
y_val = val_frame[LABEL].to_numpy(dtype=int)
val_ids = val_frame['TransactionID'].to_numpy()
X_val = baseline_transform(val_frame, baseline_bundle)
del val_frame
gc.collect()
baseline_scores = baseline_model.predict_proba(X_val)[:, 1]
baseline_ap = average_precision_score(y_val, baseline_scores)
assert np.isclose(baseline_ap, baseline_run['validation_metrics']['Average precision'], atol=1e-8, rtol=0), 'Saved baseline AP did not reproduce. Investigate input or environment changes before continuing.'

if best_id == 0:
    selected_model = baseline_model
    print('Baseline configuration won CV; retaining the Step 1 model.')
else:
    train_frame = pd.read_csv(TRAIN_PATH, low_memory=False)
    y_train = train_frame[LABEL].to_numpy(dtype=int)
    X_train = baseline_transform(train_frame, baseline_bundle)
    del train_frame
    gc.collect()
    sampler = RandomUnderSampler(sampling_strategy=SAMPLING_RATIO, random_state=SEED)
    X_sampled, y_sampled = sampler.fit_resample(X_train, y_train)
    del X_train, y_train
    gc.collect()
    selected_model = clone(baseline_model).set_params(**best_params)
    selected_model.fit(X_sampled, y_sampled)
    del X_sampled, y_sampled
    gc.collect()
selected_scores = selected_model.predict_proba(X_val)[:, 1]

def metrics_at_half(y, scores):
    pred = (scores > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {'Average precision': float(average_precision_score(y, scores)),
            'ROC AUC': float(roc_auc_score(y, scores)),
            'Precision': float(precision_score(y, pred, zero_division=0)),
            'Recall': float(recall_score(y, pred, zero_division=0)),
            'F1': float(f1_score(y, pred, zero_division=0)),
            'False positive rate': float(fp / (fp + tn)),
            'True positives': int(tp), 'False positives': int(fp),
            'False negatives': int(fn), 'True negatives': int(tn)}

baseline_metrics = metrics_at_half(y_val, baseline_scores)
selected_metrics = metrics_at_half(y_val, selected_scores)
comparison = pd.DataFrame({'Step 1 baseline': baseline_metrics, 'CV-selected model': selected_metrics})
comparison['Difference'] = comparison['CV-selected model'] - comparison['Step 1 baseline']
comparison.to_csv(OUTPUT_DIR / 'validation_comparison.csv', index_label='Metric')
display(comparison)

## 7. Model and score export

Save the fitted model, encoder, feature order, selected parameters and validation scores. Row-level validation scores support the following threshold-selection stage without retraining. Test data remain unused.


In [ ]:
metadata = {'status': 'trained_and_validated', 'selection': selection,
    'versions': runtime_versions, 'seed': SEED, 'sampling_ratio': SAMPLING_RATIO,
    'threshold': 0.5, 'threshold_rule': 'score > 0.5',
    'baseline_validation_metrics': baseline_metrics, 'selected_validation_metrics': selected_metrics,
    'test_performance_evaluated': False, 'baseline_run_reference': baseline_run,
    'cv_gap_days': GAP_DAYS, 'label_delay_days': LABEL_DELAY_DAYS,
    'final_fit_protocol': 'original full training split, same as Step 1',
    'constructor_parameters': selected_model.get_params()}
save_json(OUTPUT_DIR / 'tuning_run.json', metadata)
selected_model.save_model(OUTPUT_DIR / 'selected_xgboost.json')
(OUTPUT_DIR / 'xgboost_effective_config.json').write_text(selected_model.get_booster().save_config())
selected_bundle = dict(baseline_bundle)
selected_bundle.update(model=selected_model, threshold=0.5, threshold_rule='>', metadata=metadata)
joblib.dump(selected_bundle, OUTPUT_DIR / 'selected_bundle.joblib', compress=3)
pd.DataFrame({'TransactionID': val_ids, 'isFraud': y_val,
              'baseline_score': baseline_scores, 'selected_score': selected_scores}).to_csv(
                  OUTPUT_DIR / 'validation_scores.csv', index=False)
reloaded = joblib.load(OUTPUT_DIR / 'selected_bundle.joblib')
np.testing.assert_allclose(reloaded['model'].predict_proba(X_val.iloc[:1000])[:, 1],
                           selected_scores[:1000], rtol=0, atol=0)
print('Saved-model reload passed. Test data were not loaded.')
print('Cross-validation summary and validation comparison saved.')

## Saved experiment results

The following tables are read from the completed experiment's exported CSV and JSON files. They show fold sizes, all candidate scores, the selected parameters and the validation comparison. This reporting cell does not train a model or recreate the missing training-cell execution logs.


In [ ]:
from pathlib import Path
import json
import pandas as pd

results_dir = Path(globals().get('PROJECT_DIR', Path.cwd())) / 'step_2_outputs'
saved_folds = pd.read_csv(results_dir / 'fold_summary.csv')
saved_cv = pd.read_csv(results_dir / 'cv_summary.csv')
saved_fits = pd.read_csv(results_dir / 'cv_fit_results.csv')
saved_selection = json.loads((results_dir / 'selected_parameters.json').read_text())
saved_validation = pd.read_csv(results_dir / 'validation_comparison.csv')

print('Chronological folds')
display(saved_folds)
print(f'Completed fits: {len(saved_fits)}; configurations: {len(saved_cv)}')
display(saved_cv.drop(columns=['parameters']))
print('Selected parameters')
print(json.dumps(saved_selection, indent=2))
print('Validation comparison')
display(saved_validation)


Chronological folds


fold,fit_rows,score_rows,fit_end_seconds,score_start_seconds,score_end_seconds_exclusive,fit_fraud_rate,score_fraud_rate
1,233881,56159,5529566,6134400,7776000,0.031717,0.038391
2,291259,64375,7171185,7776000,9417600,0.033554,0.035557
3,356216,63069,8812779,9417600,11145600,0.033797,0.040368


Completed fits: 63; configurations: 21


candidate_id,mean_cv_ap,std_cv_ap,folds_completed,total_seconds
15,0.679166,0.029057,3,199.935389
7,0.678598,0.027398,3,293.813433
13,0.677716,0.034711,3,206.601025
4,0.677134,0.030615,3,256.104359
10,0.676132,0.024971,3,87.589273
16,0.675167,0.028056,3,152.086591
18,0.674808,0.030317,3,206.187703
19,0.674718,0.029179,3,103.944769
11,0.671927,0.029777,3,100.754963
9,0.671269,0.029466,3,248.301048


Selected parameters


{
  "candidate_id": 15,
  "parameters": {
    "subsample": 1.0,
    "reg_lambda": 3.0,
    "reg_alpha": 0.0,
    "n_estimators": 400,
    "min_child_weight": 3,
    "max_depth": 6,
    "learning_rate": 0.05,
    "colsample_bytree": 0.85
  },
  "mean_cv_ap": 0.679165851949903,
  "baseline_mean_cv_ap": 0.6538630432352259,
  "mean_cv_ap_gain": 0.02530280871467705,
  "selection_metric": "mean of three temporal-fold average precision scores"
}


Validation comparison


Metric,Step 1 baseline,CV-selected model,Difference
Average precision,0.705831,0.709403,0.003572
ROC AUC,0.947211,0.951365,0.004153
Precision,0.740290,0.777778,0.037488
Recall,0.595616,0.574627,-0.020989
F1,0.660119,0.660944,0.000825
False positive rate,0.007587,0.005961,-0.001626
True positives,1277.000000,1232.000000,-45.000000
False positives,448.000000,352.000000,-96.000000
False negatives,867.000000,912.000000,45.000000
True negatives,58604.000000,58700.000000,96.000000
